In [1]:
import pandas as pd
from dotenv import load_dotenv
import os
import requests
import time
import hashlib
import hmac
import plotly.express as px
import plotly.graph_objs as go
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import openpyxl



os.environ.pop("API_KEY")
os.environ.pop("SECRET_KEY")

load_dotenv()

API_KEY = os.environ["API_KEY"]
API_SECRET = os.environ["SECRET_KEY"]

In [2]:
def boilerplate(params, endpoint):
    BASE_URL = 'https://api.binance.com'

    # Create the signature using HMAC-SHA256
    query_string = '&'.join([f"{key}={value}" for key, value in params.items()])
    signature = hmac.new(
        API_SECRET.encode('utf-8'),
        query_string.encode('utf-8'),
        hashlib.sha256
    ).hexdigest()

    # Add the signature to the parameters
    params['signature'] = signature

    # Define the headers with API key
    headers = {
        'X-MBX-APIKEY': API_KEY
    }

    # Send the GET request
    response = requests.get(BASE_URL + endpoint, headers=headers, params=params)

    # Check response status and print the result
    if response.status_code == 200:
        # print(response.json())
        return response.json()
    else:
        print(f"Error: {response.status_code}, {response.text}")
        return Exception(f"Error: {response.status_code}, {response.text}")
    
def boilerplate1(params, endpoint):
    BASE_URL = 'https://api.binance.com'

    # Create the signature using HMAC-SHA256
    query_string = '&'.join([f"{key}={value}" for key, value in params.items()])
    # signature = hmac.new(
    #     API_SECRET.encode('utf-8'),
    #     query_string.encode('utf-8'),
    #     hashlib.sha256
    # ).hexdigest()

    # # Add the signature to the parameters
    # params['signature'] = signature

    # Define the headers with API key
    headers = {
        'X-MBX-APIKEY': API_KEY
    }

    # Send the GET request
    response = requests.get(BASE_URL + endpoint, headers=headers, params=params)

    # Check response status and print the result
    if response.status_code == 200:
        # print(response.json())
        return response.json()
    else:
        print(f"Error: {response.status_code}, {response.text}")
        return Exception(f"Error: {response.status_code}, {response.text}")
    

def boilerplate_post(params, endpoint):
    BASE_URL = 'https://api.binance.com'

    # Create the signature using HMAC-SHA256
    query_string = '&'.join([f"{key}={value}" for key, value in params.items()])
    signature = hmac.new(
        API_SECRET.encode('utf-8'),
        query_string.encode('utf-8'),
        hashlib.sha256
    ).hexdigest()

    # Add the signature to the parameters
    params['signature'] = signature

    # Define the headers with API key
    headers = {
        'X-MBX-APIKEY': API_KEY
    }

    # Send the POST request
    response = requests.post(BASE_URL + endpoint, headers=headers, params=params)

    # Check response status and print the result
    if response.status_code == 200:
        # print(response.json())
        return response.json()
    else:
        print(f"Error: {response.status_code}, {response.text}")
        return Exception(f"Error: {response.status_code}, {response.text}")

In [3]:
def get_DCI_products_history(order_nums):

    # Define the endpoint and base URL
    endpoint = '/sapi/v1/dci/product/positions'

    # Define request parameters
    params = {
        'pageSize':order_nums,
        'timestamp': int(time.time() * 1000)  # Current timestamp in milliseconds
    }

    return boilerplate(params, endpoint)

In [4]:
response = get_DCI_products_history(order_nums=25)


In [5]:
response

{'total': 180,
 'list': [{'id': '17503091',
   'investCoin': 'USDT',
   'exercisedCoin': 'ETH',
   'subscriptionAmount': '1317.9',
   'strikePrice': '2150',
   'duration': 4,
   'settleDate': 1776412800000,
   'purchaseStatus': 'SETTLED',
   'apr': '1.2693',
   'orderId': 47243502160,
   'purchaseEndTime': 1776355200000,
   'optionType': 'PUT',
   'autoCompoundPlan': 'NONE',
   'settlePrice': '2334.161',
   'isExercised': False,
   'settleAsset': 'USDT',
   'settleAmount': '1336.21881',
   'subscriptionTime': 1776069789377},
  {'id': '17494115',
   'investCoin': 'ETH',
   'exercisedCoin': 'USDT',
   'subscriptionAmount': '11.1244',
   'strikePrice': '2400',
   'duration': 5,
   'settleDate': 1776412800000,
   'purchaseStatus': 'SETTLED',
   'apr': '0.4332',
   'orderId': 47208303904,
   'purchaseEndTime': 1776355200000,
   'optionType': 'CALL',
   'autoCompoundPlan': 'NONE',
   'settlePrice': '2334.161',
   'isExercised': False,
   'settleAsset': 'ETH',
   'settleAmount': '11.19003396'

In [6]:
import pandas as pd

df = pd.DataFrame(response["list"])
df["Product"] = df["optionType"].map({
    "CALL": "Sell High",
    "PUT": "Buy Low"
})
df = df.rename(columns={'id': 'Product ID', "strikePrice":"Target Price", "apr":"APR"})
df["Subscription Date(UTC)"] = pd.to_datetime(df['settleDate'],unit='ms')
# subtract duration (integer days) from settleDate
df["Subscription Date(UTC)"] = df["Subscription Date(UTC)"] - pd.to_timedelta(df["duration"], unit="D")
df["subscriptionAmount"] = df["subscriptionAmount"].astype(float)
df["Subscription Amount"] = df["subscriptionAmount"].astype(str) + " " + df["investCoin"]
df["Settlement Date(UTC)"] = pd.to_datetime(df['settleDate'],unit='ms').dt.strftime('%Y-%m-%d')
df["settleAmount"] = df["settleAmount"].astype(float)
df["Settlement Amount"] = df["settleAmount"].astype(str) + " " + df["settleAsset"]

# df
# df["Target Price"] = df["strikePrice"]
# df = df.rename(columns={'id': 'Product ID'})


fin = df[[
    "Product", "Product ID", "Subscription Date(UTC)",
    "Subscription Amount", "Target Price",
    "Settlement Date(UTC)", "APR", "Settlement Amount"
]].sort_values(by="Settlement Date(UTC)", ascending=True)
fin.to_excel("data/dual_account_statement.xlsx", sheet_name="DualInvestment", index=False)


In [7]:
len(df)

25

In [8]:
df

,Product ID,investCoin,exercisedCoin,subscriptionAmount,Target Price,duration,settleDate,purchaseStatus,APR,orderId,...,settlePrice,isExercised,settleAsset,settleAmount,subscriptionTime,Product,Subscription Date(UTC),Subscription Amount,Settlement Date(UTC),Settlement Amount
0,17503091,USDT,ETH,1317.9000,2150,4,1776412800000,SETTLED,1.2693,47243502160,...,2334.161,False,USDT,1336.218810,1776069789377,Buy Low,2026-04-13 08:00:00,1317.9 USDT,2026-04-17,1336.21881 USDT
1,17494115,ETH,USDT,11.1244,2400,5,1776412800000,SETTLED,0.4332,47208303904,...,2334.161,False,ETH,11.190034,1775966524114,Sell High,2026-04-12 08:00:00,11.1244 ETH,2026-04-17,11.19003396 ETH
2,17442097,USDT,ETH,1294.6000,2150,4,1775808000000,SETTLED,1.6433,47041244454,...,2183.3535,False,USDT,1317.902800,1775470362571,Buy Low,2026-04-06 08:00:00,1294.6 USDT,2026-04-10,1317.9028 USDT
3,17440880,ETH,USDT,11.0936,2300,4,1775808000000,SETTLED,0.2486,47039944467,...,2183.3535,False,ETH,11.123553,1775466306102,Sell High,2026-04-06 08:00:00,11.0936 ETH,2026-04-10,11.12355272 ETH
4,17400103,ETH,USDT,11.0729,2275,2,1775203200000,SETTLED,0.3296,46907817452,...,2067.4895,False,ETH,11.092831,1775032690220,Sell High,2026-04-01 08:00:00,11.0729 ETH,2026-04-03,11.09283122 ETH
5,17240086,USDT,ETH,1268.9000,1975,12,1774598400000,SETTLED,0.6168,46368609792,...,2068.36566667,False,USDT,1294.531780,1773556023716,Buy Low,2026-03-15 08:00:00,1268.9 USDT,2026-03-27,1294.53178 USDT
6,17240083,ETH,USDT,10.9970,2450,12,1774598400000,SETTLED,0.2092,46368609987,...,2068.36566667,False,ETH,11.071780,1773555951366,Sell High,2026-03-15 08:00:00,10.997 ETH,2026-03-27,11.0717796 ETH
7,17194638,ETH,USDT,5.0145,2200,3,1773388800000,SETTLED,0.3513,46192028559,...,2098.44966667,False,ETH,5.028541,1773116457741,Sell High,2026-03-10 08:00:00,5.0145 ETH,2026-03-13,5.0285406 ETH
8,17167428,USDT,ETH,1246.4000,1975,7,1773388800000,SETTLED,0.9472,46073827946,...,2098.44966667,False,USDT,1268.959840,1772791784308,Buy Low,2026-03-06 08:00:00,1246.4 USDT,2026-03-13,1268.95984 USDT
9,17156845,ETH,USDT,5.9329,2400,8,1773388800000,SETTLED,0.2769,46037791554,...,2098.44966667,False,ETH,5.968497,1772706385796,Sell High,2026-03-05 08:00:00,5.9329 ETH,2026-03-13,5.9684974 ETH
